# Домашнее задание №3

**Цель:** добиться качества 0.9 на top-1 accuracy (и 0.98 на top-5) при ограничении **400к обучаемых параметров** на **уникальном** сабсете из [cоревнования на Kaggle](https://www.kaggle.com/competitions/dog-breed-identification/data)

Для выполнения нам понадобятся данные из соревнования по ссылке, **мы работаем только с частью "train"**

### Система оценки:

#### Подход к решению (макс 5 баллов):

**1 балл** - рассмотрено 1 решение по transfer learning одной модели, взяты стандартный transform / loss / optimizer  
**3 балла** - рассмотренно минимум 2 решения (разные модели / разная заморозка слоев / разный head) или присутсвуют 2-3 эксперимента с оптимизацией (шедуллинг, оптимайзер, необычные аугментации, игры с лоссом)  
**5 баллов** - рассмотренно минимум 2 решения (разные модели, разная заморозка слоев) + присутсвуют 2-3 эксперимента с оптимизацией (шедуллинг, оптимайзер, необычные аугментации, игры с лоссом)  
> Промежуточные оценки выставляются по широте экспериментов и доказательной базе к их применению

#### Качество решения на тестовой выборке (макс 5 баллов), накопительная система:
**+ 3 балла** за превышение 0.9 на top-1 accuracy  
**+ 2 балла** за превышение 0.98 на top-3 accuracy
> Может быть накинуты баллы даже при отсутсвии метрик выше за доказательство недостижимости метрики на конкретном датасете (все переборы не проверялись)

**! Дополнительным баллом можно поощрить оригинальность и проработанность решения**

## 1. Необходимые импорты и вспомогательные функции

In [ ]:
import pandas as pd
import torch
import random
import numpy as np
import os
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import time
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms, models, datasets
from torch.utils.data import Dataset, DataLoader
from PIL import Image

%matplotlib inline

In [ ]:
def set_seed_to_all(seed: int) -> None:
    """
    Проставить всему сид. 
    Важно: не влияет на детерменированность обучения.
    """
    # Python
    random.seed(seed)
    
    # NumPy
    np.random.seed(seed)
    
    # PyTorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # Python hash seed
    os.environ["PYTHONHASHSEED"] = str(seed)


def train_test_val_split(df: pd.DataFrame, seed: int) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Деление данных на трейн/тест/валидацию.
    """
    # Train vs valtest
    train_df, temp_df = train_test_split(
        df,
        test_size=0.3,
        stratify=df["breed"],
        random_state=seed
    )
    
    # Val vs test
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.5,
        stratify=temp_df["breed"],
        random_state=seed
    )

    print("Train:", len(train_df))
    print("Val:", len(val_df))
    print("Test:", len(test_df))
    return train_df, val_df, test_df


def plot_distribution(df: pd.DataFrame, title: str) -> None:
    """
    Отрисовка классов.
    """
    counts = df["breed"].value_counts().sort_values(ascending=False)
    plt.figure()
    counts.plot(kind="bar")
    plt.title(title)
    plt.xlabel("Breed")
    plt.ylabel("Number of images")
    plt.xticks(rotation=90)
    plt.tight_layout()


class DogDataset(Dataset):
    """
    Вариант датасета.
    """
    def __init__(
        self,
        df: pd.DataFrame,
        images_dir: str = "dog-breed-identification/train",
        images_ext = ".jpg",
        transform: transforms.Compose | None = None,
        class_to_idx: dict[str, int] | None = None
    ) -> None:
        self.df = df
        self.transform = transform
        self.images_dir = images_dir
        self.images_ext = images_ext

        if class_to_idx is None:
            classes = sorted(df["breed"].unique())
            self.class_to_idx = {cls: i for i, cls in enumerate(classes)}
        else:
            self.class_to_idx = class_to_idx

        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}

    def __len__(self) -> None:
        return len(self.df)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, int]:
        row = self.df.iloc[idx]

        img_path = os.path.join(self.images_dir, row["id"] + self.images_ext)
        label_str = row["breed"]
        label = self.class_to_idx[label_str]

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label


@torch.no_grad()
def evaluate_topk_with_time(
    model: nn.Module,
    dataloader: DataLoader,
    device: str | torch.device,
    k: tuple[int, ...] = (1, 3)
) -> dict[str, float]:
    """
    Целевая метрика.
    """
    model.eval()
    correct = {kk: 0 for kk in k}
    total = 0

    start = time.time()

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, pred = outputs.topk(max(k), dim=1)

        total += labels.size(0)

        for kk in k:
            correct_k = (pred[:, :kk] == labels.unsqueeze(1)).any(dim=1).sum().item()
            correct[kk] += correct_k

    elapsed = time.time() - start
    results = {f"top{kk}": correct[kk] / total for kk in k}
    results["inference_time_sec"] = elapsed
    results["samples_per_sec"] = total / elapsed

    return results

## 2. Создание УНИКАЛЬНОГО датасета

In [ ]:
DATE_OF_BIRTH: int =  # ВСТАВИТЬ СВОЙ ДЕНЬ РОЖДЕНИЯ 
MONTH_OF_BIRTH: int =  # ВСТАВИТЬ СВОЙ МЕСЯЦ РОЖДЕНИЯ 
HEIGHT =  # ВСТАВИТЬ СВОЙ РОСТ
LASTNAME =  # ВСТАВИТЬ СВОЮ ФАМИЛИЮ

# Получем сид по формуле
seed = (DATE_OF_BIRTH + MONTH_OF_BIRTH) * (ord(LASTNAME[0])%(HEIGHT%10))
set_seed_to_all(seed)

In [ ]:
labels_df = pd.read_csv("dog-breed-identification/labels.csv")
label_names = list(set(labels_df.breed.values))
classes = random.sample(label_names, 20)
my_data = labels_df[labels_df["breed"].isin(classes)]

In [ ]:
train_df, val_df, test_df = train_test_val_split(my_data, seed)

In [ ]:
plot_distribution(train_df, "Train Label Distribution")
plot_distribution(val_df, "Validation Label Distribution")
plot_distribution(test_df, "Test Label Distribution")

## 3. Творческая часть

In [ ]:
## TODO: Грузим модель и издеваемся над ней

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert trainable_params < 400000, "Number of params must be less than 400.000"

In [ ]:
## Создаем датасета, лоадеры, придумываем лосс и оптимиацию

In [ ]:
## Проводим итерации трейна

## 4. Проверяем метрики и условия

In [ ]:
result = evaluate_topk_with_time(model, test_loader, device)
assert trainable_params < 400000, "Number of params must be less than 400.000"
if result['top1'] > 0.9:
    print("Добились метрики на top-1")
if result['top3'] > 0.98:
    print("Добились метрики на top-3")
print("Резульат: ", result)